# Rol 2 - Síntesis y fitness

Notebook inicial. Ejecutar desde un entorno limpio.
Reutilizar las funciones compartidas de src.


In [3]:
%pip install soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 4.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import sys

import numpy as np
from IPython.display import Audio, display

# Funciona al iniciar Jupyter desde la raíz o desde notebooks/.
RAIZ = Path.cwd().resolve()

if not (RAIZ / "src" / "sintesis.py").is_file():
    RAIZ = RAIZ.parent

if not (RAIZ / "src" / "sintesis.py").is_file():
    raise FileNotFoundError(
        "Abre el notebook desde la raíz del proyecto "
        "o desde su carpeta notebooks/."
    )

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.sintesis import (
    FRECUENCIA_MUESTREO,
    sintetizar_instrumento,
    guardar_wav,
)

In [5]:
# Sol5, pausa, La5, pausa, Si5, pausa, La5, pausa, Sol5.
notas = np.array(
    [79, -1, 81, -1, 83, -1, 81, -1, 79],
    dtype=int,
)

duraciones = np.array(
    [0.6, 0.1, 0.6, 0.1, 0.6, 0.1, 0.6, 0.1, 0.6],
    dtype=float,
)

# Valores de demostración; todavía no fueron optimizados por el AG.
gen_prueba = np.array(
    [0.8, 0.2, 0.1, 0.2, 0.4],
    dtype=float,
)

In [6]:
audio = sintetizar_instrumento(
    notas,
    duraciones,
    gen_prueba,
)

print(f"Frecuencia de muestreo: {FRECUENCIA_MUESTREO} Hz")
print(f"Duración: {len(audio) / FRECUENCIA_MUESTREO:.2f} segundos")
print(f"Amplitud máxima: {np.max(np.abs(audio)):.4f}")

display(
    Audio(
        audio,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    )
)

Frecuencia de muestreo: 22050 Hz
Duración: 3.40 segundos
Amplitud máxima: 0.4491


In [7]:
ruta_audio = guardar_wav(
    RAIZ / "resultados" / "audios" / "prueba_sintesis_rol2.wav",
    audio,
)

print(f"Audio guardado: {ruta_audio.relative_to(RAIZ)}")

Audio guardado: resultados\audios\prueba_sintesis_rol2.wav


In [8]:
from src.fitness import (
    evaluar_audio,
    evaluar_individuo,
)

In [9]:
resultado_identico = evaluar_audio(audio, audio)

print("Comparación del audio consigo mismo:")
print(resultado_identico)

assert np.isclose(
    resultado_identico["error_stft_db"], 0.0
)

assert np.isclose(
    resultado_identico["fitness_stft"], 1.0
)

assert np.isclose(
    resultado_identico["fitness_croma"], 1.0
)

Comparación del audio consigo mismo:
{'error_stft_db': 0.0, 'fitness_stft': 1.0, 'fitness_croma': 1.0}


In [10]:
gen_distinto = np.array(
    [0.1, 0.9, 0.9, 0.9, 0.1],
    dtype=float,
)

audio_distinto = sintetizar_instrumento(
    notas,
    duraciones,
    gen_distinto,
)

resultado_distinto = evaluar_audio(
    audio_distinto,
    audio,
)

print("Comparación entre dos sonidos sintéticos:")
print(
    f"Error STFT: "
    f"{resultado_distinto['error_stft_db']:.4f} dB"
)
print(
    f"Fitness STFT: "
    f"{resultado_distinto['fitness_stft']:.4f}"
)
print(
    f"Fitness croma: "
    f"{resultado_distinto['fitness_croma']:.4f}"
)

assert resultado_distinto["error_stft_db"] > 0.0
assert resultado_distinto["fitness_stft"] < 1.0

print("\nSonido inicial:")
display(
    Audio(
        audio,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    )
)

print("Sonido con otros parámetros:")
display(
    Audio(
        audio_distinto,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    )
)

Comparación entre dos sonidos sintéticos:
Error STFT: 0.3560 dB
Fitness STFT: 0.9656
Fitness croma: 0.9853

Sonido inicial:


Sonido con otros parámetros:


In [11]:
aptitud = evaluar_individuo(
    gen=gen_distinto,
    notas=notas,
    duraciones=duraciones,
    referencia=audio,  # Solo para esta comprobación técnica.
    metrica="stft",
)

print(f"Aptitud que recibiría el AG: {aptitud:.4f}")

Aptitud que recibiría el AG: 0.9656


## Función de aptitud

La métrica principal compara las magnitudes de los espectrogramas
STFT expresadas en decibelios. Ambos audios deben estar alineados
y tener la misma frecuencia de muestreo y longitud.

Se calcula el error absoluto medio entre espectrogramas y se
transforma en aptitud:

fitness = 1 / (1 + error_db / 10)

El AG maximiza esta aptitud. Una coincidencia exacta obtiene 1.
El factor 10 controla la escala de la transformación; no representa
un umbral validado de calidad perceptual.

La segunda métrica compara cromas mediante similitud coseno.
Permite comprobar coincidencias de altura musical, aunque puede
distinguir poco entre timbres que interpretan las mismas notas.
Por ello, se evaluará como alternativa y se mantendrá STFT como
métrica principal inicial.

No se utiliza MSE de la onda cruda porque pequeñas diferencias de
fase pueden producir errores grandes incluso entre sonidos
perceptualmente similares.

Las comparaciones iniciales utilizan sonidos del sintetizador
para verificar las funciones. No demuestran semejanza con una
flauta real ni constituyen resultados del algoritmo genético.